# Text Preprocessing and Data Splitting

Clean review text and create train/validation/test splits for model training.

**Preprocessing Steps**:
- Lowercase text
- Remove punctuation and special characters
- Remove English stopwords
- Lemmatization for word normalization

**Data Splitting**:
- Stratified train/validation/test split (60/20/20)
- Maintains label distribution across splits

## Imports

In [1]:
import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
import string
import os
from io import BytesIO
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

print('All imports successful')

All imports successful


## Helper Classes

In [2]:
class TextPreprocessor:
    """Text preprocessing and data splitting for sentiment analysis."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3', region_name='us-east-1')
        self.df_data = None
        self.df_train = None
        self.df_val = None
        self.df_test = None
        self.lemmatizer = WordNetLemmatizer()
        self.set_stopwords = set(stopwords.words('english'))
    
    def import_data(self):
        """Load cleaned data from S3."""
        str_uri = f's3://{self.str_bucket}/00_data_collection/employee_reviews_clean.parquet'
        self.df_data = pd.read_parquet(str_uri)
        print(f'Loaded {len(self.df_data):,} reviews from S3')
        return True
    
    def clean_text(self, str_text):
        """Clean and preprocess a single text string."""
        # Convert to lowercase
        str_text = str_text.lower()
        
        # Remove URLs (if any)
        str_text = ' '.join([word for word in str_text.split() if not word.startswith('http')])
        
        # Remove punctuation
        str_text = str_text.translate(str.maketrans('', '', string.punctuation))
        
        # Split into tokens
        list_tokens = str_text.split()
        
        # Remove stopwords and lemmatize
        list_tokens = [self.lemmatizer.lemmatize(word) 
                       for word in list_tokens 
                       if word not in self.set_stopwords and len(word) > 2]
        
        return ' '.join(list_tokens)
    
    def preprocess_all_data(self):
        """Apply text cleaning to entire dataset."""
        print('\nPreprocessing all reviews...')
        list_cleaned = []
        
        for str_text in tqdm(self.df_data['review_text'], desc='Cleaning text'):
            str_cleaned = self.clean_text(str_text)
            list_cleaned.append(str_cleaned)
        
        self.df_data['review_text_clean'] = list_cleaned
        print(f'Preprocessed {len(self.df_data)} reviews')
    
    def split_data(self, flt_train_size=0.6, flt_val_size=0.2, int_random_state=42):
        """Create stratified train/validation/test split."""
        # First split: 60% train, 40% temp
        self.df_train, df_temp = train_test_split(
            self.df_data,
            train_size=flt_train_size,
            stratify=self.df_data['sentiment_3class'],
            random_state=int_random_state
        )
        
        # Second split: split temp into 50/50 for validation and test
        # 20% of total = 50% of 40% remaining
        self.df_val, self.df_test = train_test_split(
            df_temp,
            train_size=0.5,
            stratify=df_temp['sentiment_3class'],
            random_state=int_random_state
        )
        
        print(f'\nData split:')
        print(f'  Train: {len(self.df_train):,} ({len(self.df_train)/len(self.df_data)*100:.1f}%)')
        print(f'  Validation: {len(self.df_val):,} ({len(self.df_val)/len(self.df_data)*100:.1f}%)')
        print(f'  Test: {len(self.df_test):,} ({len(self.df_test)/len(self.df_data)*100:.1f}%)')
    
    def print_split_statistics(self):
        """Print class distribution in each split."""
        list_labels = ['Negative', 'Neutral', 'Positive']
        list_dfs = [('Train', self.df_train), ('Validation', self.df_val), ('Test', self.df_test)]
        
        print('\n' + '='*70)
        print('CLASS DISTRIBUTION IN DATA SPLITS')
        print('='*70)
        
        for str_split_name, df_split in list_dfs:
            print(f'\n{str_split_name} Set (n={len(df_split):,})')
            print('-' * 70)
            for int_label, str_label_name in enumerate(list_labels):
                int_count = (df_split['sentiment_3class'] == int_label).sum()
                flt_pct = (int_count / len(df_split)) * 100
                print(f'  {str_label_name:10s}: {int_count:6,} ({flt_pct:5.1f}%)')
            
            # Binary distribution
            print(f'\n  Binary labels:')
            list_binary_labels = ['Negative', 'Positive']
            for int_label, str_label_name in enumerate(list_binary_labels):
                int_count = (df_split['sentiment_binary'] == int_label).sum()
                flt_pct = (int_count / len(df_split)) * 100
                print(f'    {str_label_name:10s}: {int_count:6,} ({flt_pct:5.1f}%)')
    
    def save_splits(self):
        """Save train/validation/test splits as parquet to S3."""
        dict_splits = {
            'train': (self.df_train, 'train_split.parquet'),
            'validation': (self.df_val, 'validation_split.parquet'),
            'test': (self.df_test, 'test_split.parquet')
        }
        
        print('\nSaving splits to S3...')
        for str_split_name, (df_split, str_filename) in dict_splits.items():
            buffer = BytesIO()
            df_split.to_parquet(buffer, index=False, engine='pyarrow')
            buffer.seek(0)
            str_s3_key = f'02_preprocessing/{str_filename}'
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue(),
                ContentType='application/octet-stream'
            )
            print(f'  {str_split_name}: s3://{self.str_bucket}/{str_s3_key}')

## Constants

In [3]:
# S3 Configuration
str_bucket = 'nlp-sentiment-analysis-demo'
str_task = '02_preprocessing'
str_dirname_output = './output'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
    print(f'✓ Created output directory: {str_dirname_output}')
except FileExistsError:
    print(f'✓ Output directory already exists: {str_dirname_output}')

✓ Output directory already exists: ./output


## Load and Preprocess Data

In [5]:
# Initialize preprocessor
cls_preprocessor = TextPreprocessor(str_bucket, str_dirname_output)

# Load data
cls_preprocessor.import_data()

Loaded 30,281 reviews from S3


True

In [6]:
# Preprocess all text
cls_preprocessor.preprocess_all_data()


Preprocessing all reviews...


Cleaning text: 100%|██████████| 30281/30281 [00:08<00:00, 3671.83it/s]

Preprocessed 30281 reviews


In [7]:
# Display sample before and after preprocessing
print('\nSample Preprocessing Results:')
print('='*100)

for int_i in range(min(3, len(cls_preprocessor.df_data))):
    print(f'\nReview {int_i + 1}:')
    print(f'  Original: {cls_preprocessor.df_data.iloc[int_i]["review_text"][:150]}...')
    print(f'  Cleaned:  {cls_preprocessor.df_data.iloc[int_i]["review_text_clean"][:150]}...')
    print(f'  Rating: {cls_preprocessor.df_data.iloc[int_i]["overall_rating"]}')
    print(f'  3-Class Label: {cls_preprocessor.df_data.iloc[int_i]["sentiment_3class"]}')


Sample Preprocessing Results:

Review 1:
  Original: People are smart and friendly Bureaucracy is slowing things down Best Company to work for...
  Cleaned:  people smart friendly bureaucracy slowing thing best company work...
  Rating: 5
  3-Class Label: 2

Review 2:
  Original: 1) Food, food, food. 15+ cafes on main campus (MTV) alone. Mini-kitchens, snacks, drinks, free breakfast/lunch/dinner, all day, errr'day.  2) Benefits...
  Cleaned:  food food food cafe main campus mtv alone minikitchens snack drink free breakfastlunchdinner day errrday benefitsperks free 247 gym access mtv campus ...
  Rating: 5
  3-Class Label: 2

Review 3:
  Original: * If you're a software engineer, you're among the kings of the hill at Google. It's an engineer-driven company without a doubt (that *is* changing, bu...
  Cleaned:  youre software engineer youre among king hill google engineerdriven company without doubt changing still engineerfocused perk amazing yes free breakfa...
  Rating: 5
  3-Class La

## Create Stratified Train/Validation/Test Split

In [8]:
# Split data
cls_preprocessor.split_data(flt_train_size=0.6, flt_val_size=0.2, int_random_state=42)


Data split:
  Train: 18,168 (60.0%)
  Validation: 6,056 (20.0%)
  Test: 6,057 (20.0%)


## Verify Split Statistics

In [9]:
# Print detailed statistics
cls_preprocessor.print_split_statistics()


CLASS DISTRIBUTION IN DATA SPLITS

Train Set (n=18,168)
----------------------------------------------------------------------
  Negative  :  2,490 ( 13.7%)
  Neutral   :  5,696 ( 31.4%)
  Positive  :  9,982 ( 54.9%)

  Binary labels:
    Negative  :  8,186 ( 45.1%)
    Positive  :  9,982 ( 54.9%)

Validation Set (n=6,056)
----------------------------------------------------------------------
  Negative  :    830 ( 13.7%)
  Neutral   :  1,898 ( 31.3%)
  Positive  :  3,328 ( 55.0%)

  Binary labels:
    Negative  :  2,728 ( 45.0%)
    Positive  :  3,328 ( 55.0%)

Test Set (n=6,057)
----------------------------------------------------------------------
  Negative  :    830 ( 13.7%)
  Neutral   :  1,899 ( 31.4%)
  Positive  :  3,328 ( 54.9%)

  Binary labels:
    Negative  :  2,729 ( 45.1%)
    Positive  :  3,328 ( 54.9%)


## Save Splits to S3

In [10]:
cls_preprocessor.save_splits()


Saving splits to S3...
  train: s3://nlp-sentiment-analysis-demo/02_preprocessing/train_split.parquet
  validation: s3://nlp-sentiment-analysis-demo/02_preprocessing/validation_split.parquet
  test: s3://nlp-sentiment-analysis-demo/02_preprocessing/test_split.parquet


## Preprocessing Summary

In [11]:
print('\n' + '='*70)
print('PREPROCESSING COMPLETE')
print('='*70)
print(f'\nOriginal dataset size: {len(cls_preprocessor.df_data):,} reviews')
print(f'\nText Preprocessing:')
print(f'  ✓ Lowercase conversion')
print(f'  ✓ Punctuation removal')
print(f'  ✓ Stopword removal')
print(f'  ✓ Lemmatization')
print(f'\nData Splits Created:')
print(f'  ✓ Training set: {len(cls_preprocessor.df_train):,} reviews')
print(f'  ✓ Validation set: {len(cls_preprocessor.df_val):,} reviews')
print(f'  ✓ Test set: {len(cls_preprocessor.df_test):,} reviews')
print(f'\nStratification: Applied to maintain class distribution')
print(f'\nOutput Location: s3://{str_bucket}/02_preprocessing/')
print('='*70)


PREPROCESSING COMPLETE

Original dataset size: 30,281 reviews

Text Preprocessing:
  ✓ Lowercase conversion
  ✓ Punctuation removal
  ✓ Stopword removal
  ✓ Lemmatization

Data Splits Created:
  ✓ Training set: 18,168 reviews
  ✓ Validation set: 6,056 reviews
  ✓ Test set: 6,057 reviews

Stratification: Applied to maintain class distribution

Output Location: s3://nlp-sentiment-analysis-demo/02_preprocessing/
